In [ ]:
# ============================================================
# 3-FOLD CROSS-VALIDATION ENSEMBLE
# DenseNet121 (4ch + 3ch)
# Deterministic Mask-Guided Tiling
# Goal: Push Kaggle F1 > 0.40
# ============================================================

import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms.functional as TF
from torchvision.models import densenet121, DenseNet121_Weights

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, confusion_matrix
import matplotlib.pyplot as plt

from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.nn.utils import clip_grad_norm_
from torchvision.transforms import ColorJitter, GaussianBlur, RandomErasing



# ============================================================
# CONFIGURATION
# ============================================================

TRAIN_IMG_DIR    = "/kaggle/input/dataset/train_images"
TRAIN_MASK_DIR   = "/kaggle/input/dataset/train_masks"
TEST_IMG_DIR     = "/kaggle/input/dataset/test_images"
TEST_MASK_DIR    = "/kaggle/input/dataset/test_masks"
TRAIN_LABELS_CSV = "/kaggle/input/dataset/train_labels.csv"

IMG_SIZE = 384
TILE_SIZE = 448
TILE_OVERLAP = 0.5
ROI_PAD = 0.15   # enlarge ROI by 15%
TRAIN_TILES_PER_SLIDE = 3
VAL_TILES_PER_SLIDE = 8
TEST_TILES_PER_SLIDE = 8

NUM_FOLDS = 3
NUM_EPOCHS = 12
PATIENCE = 5

BASE_LR_BACKBONE = 3e-5
BASE_LR_HEAD     = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05

BATCH_SIZE = 8
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# ============================================================
# SEEDING
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(SEED)



# ============================================================
# DATA LOADING
# ============================================================

df = pd.read_csv(TRAIN_LABELS_CSV)
label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["label"])

class_names = list(label_encoder.classes_)
num_classes = len(class_names)



# ============================================================
# MASK UTILITIES
# ============================================================

def load_image(path):
    return Image.open(path).convert("RGB")

def load_mask(path, size_fallback=None):
    if os.path.exists(path):
        m = Image.open(path).convert("L")
    else:
        if size_fallback is None:
            raise ValueError("Mask missing & no fallback size given")
        m = Image.new("L", size_fallback, 255)
    return m

def get_mask_bbox(mask_np):
    ys, xs = np.where(mask_np > 0)
    if len(xs) == 0:
        return None
    return xs.min(), ys.min(), xs.max(), ys.max()

def expand_roi(x_min, y_min, x_max, y_max, w, h, ratio=0.15):
    pad_x = int((x_max - x_min) * ratio)
    pad_y = int((y_max - y_min) * ratio)
    x_min = max(0, x_min - pad_x)
    y_min = max(0, y_min - pad_y)
    x_max = min(w - 1, x_max + pad_x)
    y_max = min(h - 1, y_max + pad_y)
    return x_min, y_min, x_max, y_max



# ============================================================
# DETERMINISTIC TILING
# ============================================================

def compute_ranked_tiles(img, mask, tile_size, overlap, max_tiles, roi_pad=0.15):
    w, h = img.size
    mask_np = np.array(mask)

    bbox = get_mask_bbox(mask_np)
    if bbox is None:
        x_min, y_min, x_max, y_max = 0, 0, w - 1, h - 1
    else:
        x_min, y_min, x_max, y_max = bbox
        x_min, y_min, x_max, y_max = expand_roi(x_min, y_min, x_max, y_max, w, h, roi_pad)

    tile_size = min(tile_size, w, h)
    stride = int(tile_size * (1 - overlap))
    stride = max(stride, tile_size // 4, 1)

    tiles = []
    mask_fracs = []

    if (x_max - x_min + 1) <= tile_size and (y_max - y_min + 1) <= tile_size:
        cx = (x_min + x_max) // 2
        cy = (y_min + y_max) // 2
        left = cx - tile_size//2
        top  = cy - tile_size//2

        left = max(0, left)
        top  = max(0, top)
        right = min(w, left + tile_size)
        bottom = min(h, top + tile_size)

        tiles.append((left, top, right, bottom))
        mask_fracs.append(np.mean(np.array(mask.crop((left, top, right, bottom))) > 0))
    else:
        x_starts = list(range(x_min, max(x_max - tile_size + 1, x_min) + 1, stride))
        y_starts = list(range(y_min, max(y_max - tile_size + 1, y_min) + 1, stride))

        if not x_starts: x_starts = [x_min]
        if not y_starts: y_starts = [y_min]

        for xs in x_starts:
            for ys in y_starts:

                left = xs
                top  = ys
                right = min(w, xs + tile_size)
                bottom = min(h, ys + tile_size)

                left = max(0, right - tile_size)
                top  = max(0, bottom - tile_size)

                tiles.append((left, top, right, bottom))
                mask_fracs.append(np.mean(np.array(mask.crop((left, top, right, bottom))) > 0))

    mask_fracs = np.array(mask_fracs)
    idx_sorted = np.argsort(-mask_fracs)
    return [tiles[i] for i in idx_sorted[:max_tiles]]



# ============================================================
# AUGMENTATIONS
# ============================================================

cj = ColorJitter(0.2, 0.2, 0.2, 0.02)
gb = GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))
re = RandomErasing(p=0.12, scale=(0.02, 0.05), ratio=(0.3, 3.0))

def transform_train(img, mask, three_channel=False):
    # Only 0/90/180/270 degree rotations
    k = random.choice([0,1,2,3])
    if k > 0:
        img = img.rotate(90*k, expand=True)
        mask = mask.rotate(90*k, expand=True)

    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT)

    img = cj(img)
    if random.random() < 0.2:
        img = gb(img)

    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    mask = mask.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)

    img_t = TF.to_tensor(img)
    mask_t = (TF.to_tensor(mask) > 0.5).float()

    if three_channel:
        img_t = TF.normalize(img_t, IMAGENET_MEAN, IMAGENET_STD)
        img_t = img_t * (0.5 + 0.5 * mask_t)  # mask emphasis
        img_t = re(img_t)
        return img_t
    else:
        img_t = TF.normalize(img_t, IMAGENET_MEAN, IMAGENET_STD)
        img_t = re(img_t)
        return torch.cat([img_t, mask_t], dim=0)  # 4ch


def transform_eval(img, mask, three_channel=False):
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    mask = mask.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    img_t = TF.to_tensor(img)
    mask_t = (TF.to_tensor(mask) > 0.5).float()

    img_t = TF.normalize(img_t, IMAGENET_MEAN, IMAGENET_STD)

    if three_channel:
        return img_t * (0.5 + 0.5 * mask_t)
    else:
        return torch.cat([img_t, mask_t], dim=0)



# ============================================================
# DATASET (INTERMEDIATE-TILE PRECOMPUTATION)
# ============================================================

class TileDataset(Dataset):
    def __init__(self, rows, img_dir, mask_dir, num_tiles=3, three_channel=False):
        self.tile_entries = []
        self.three_channel = three_channel

        print("Precomputing tiles...")
        for idx in tqdm(range(len(rows))):
            r = rows.iloc[idx]
            fname = r["sample_index"]
            label = int(r["label_encoded"])

            img_path = os.path.join(img_dir, fname)
            mask_path = os.path.join(mask_dir, fname)

            img = load_image(img_path)
            mask = load_mask(mask_path, img.size)

            bboxes = compute_ranked_tiles(
                img, mask,
                tile_size=TILE_SIZE,
                overlap=TILE_OVERLAP,
                max_tiles=num_tiles,
                roi_pad=ROI_PAD
            )

            for (l,t,rgt,btm) in bboxes:
                self.tile_entries.append({
                    "img_path": img_path,
                    "mask_path": mask_path,
                    "label": label,
                    "bbox": (l,t,rgt,btm),
                })

        print("Total tiles:", len(self.tile_entries))

    def __len__(self):
        return len(self.tile_entries)

    def __getitem__(self, idx):
        entry = self.tile_entries[idx]
        img = load_image(entry["img_path"])
        mask = load_mask(entry["mask_path"], img.size)

        l,t,rgt,btm = entry["bbox"]
        img_tile = img.crop((l,t,rgt,btm))
        mask_tile = mask.crop((l,t,rgt,btm))

        x = transform_train(img_tile, mask_tile, three_channel=self.three_channel)
        y = entry["label"]
        return x, y



# ============================================================
# MODEL WRAPPERS (3ch + 4ch DenseNet121)
# ============================================================

class DenseNet121_4ch(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        conv = base.features.conv0
        new_conv = nn.Conv2d(
            4, conv.out_channels, kernel_size=conv.kernel_size,
            stride=conv.stride, padding=conv.padding, bias=False
        )
        with torch.no_grad():
            new_conv.weight[:, :3] = conv.weight
            new_conv.weight[:, 3:4] = conv.weight.mean(dim=1, keepdim=True)
        base.features.conv0 = new_conv
        base.classifier = nn.Linear(base.classifier.in_features, num_classes)
        self.model = base
    def forward(self, x):
        return self.model(x)


class DenseNet121_3ch(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        base.classifier = nn.Linear(base.classifier.in_features, num_classes)
        self.model = base
    def forward(self, x):
        return self.model(x)



# ============================================================
# TRAINING FUNCTIONS
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    losses = []
    preds_all = []
    labels_all = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        losses.append(loss.item())
        preds_all.extend(logits.argmax(1).detach().cpu().numpy())
        labels_all.extend(y.cpu().numpy())

    return np.mean(losses), f1_score(labels_all, preds_all, average="macro")



# ============================================================
# VALIDATION (SLIDE-LEVEL)
# ============================================================

def evaluate_slides(model, rows, img_dir, mask_dir, num_tiles, three_channel=False):
    model.eval()
    losses = []
    preds_all = []
    labels_all = []

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    with torch.no_grad():
        for i in tqdm(range(len(rows)), desc="Validating"):
            r = rows.iloc[i]
            fname = r["sample_index"]
            label = int(r["label_encoded"])

            img_path = os.path.join(img_dir, fname)
            mask_path = os.path.join(mask_dir, fname)

            img = load_image(img_path)
            mask = load_mask(mask_path, img.size)

            bboxes = compute_ranked_tiles(
                img, mask,
                tile_size=TILE_SIZE,
                overlap=TILE_OVERLAP,
                max_tiles=num_tiles,
                roi_pad=ROI_PAD
            )

            tile_logits = []

            for (l,t,rgt,btm) in bboxes:
                tile_img = img.crop((l,t,rgt,btm))
                tile_mask = mask.crop((l,t,rgt,btm))
                x = transform_eval(tile_img, tile_mask, three_channel=three_channel).unsqueeze(0).to(DEVICE)
                logits = model(x)
                tile_logits.append(logits)

            slide_logits = torch.mean(torch.stack(tile_logits), dim=0)
            loss = criterion(slide_logits, torch.tensor([label], device=DEVICE))
            losses.append(loss.item())

            pred = slide_logits.argmax(1).item()
            preds_all.append(pred)
            labels_all.append(label)

    return np.mean(losses), f1_score(labels_all, preds_all, average="macro"), np.array(labels_all), np.array(preds_all)



# ============================================================
# K-FOLD CROSS VALIDATION TRAINING
# ============================================================

skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
fold_models_4ch = []
fold_models_3ch = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df["label_encoded"])):
    print(f"\n===== FOLD {fold+1}/{NUM_FOLDS} =====")

    train_rows = df.iloc[train_idx]
    val_rows   = df.iloc[val_idx]

    # Tile datasets
    train_ds_4ch = TileDataset(train_rows, TRAIN_IMG_DIR, TRAIN_MASK_DIR,
                               num_tiles=TRAIN_TILES_PER_SLIDE, three_channel=False)
    train_ds_3ch = TileDataset(train_rows, TRAIN_IMG_DIR, TRAIN_MASK_DIR,
                               num_tiles=TRAIN_TILES_PER_SLIDE, three_channel=True)

    train_loader_4ch = DataLoader(train_ds_4ch, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    train_loader_3ch = DataLoader(train_ds_3ch, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    # MODELS
    m4 = DenseNet121_4ch(num_classes).to(DEVICE)
    m3 = DenseNet121_3ch(num_classes).to(DEVICE)

    # TRAIN 4ch model
    optimizer = optim.AdamW([
        {"params": [p for n,p in m4.named_parameters() if "classifier" not in n], "lr":BASE_LR_BACKBONE},
        {"params": m4.model.classifier.parameters(), "lr":BASE_LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

    best_f1 = 0
    patience = 0
    for epoch in range(NUM_EPOCHS):
        print(f"\nFOLD {fold+1} – 4ch – EPOCH {epoch+1}")

        tr_loss, tr_f1 = train_one_epoch(m4, train_loader_4ch, optimizer, criterion)
        val_loss, val_f1, _, _ = evaluate_slides(
            m4, val_rows, TRAIN_IMG_DIR, TRAIN_MASK_DIR,
            num_tiles=VAL_TILES_PER_SLIDE, three_channel=False
        )
        print(f"Train F1={tr_f1:.4f}  Val F1={val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            patience = 0
            torch.save(m4.state_dict(), f"best_fold{fold}_4ch.pth")
        else:
            patience += 1
            if patience >= PATIENCE:
                break
        scheduler.step()

    fold_models_4ch.append(f"best_fold{fold}_4ch.pth")

    # TRAIN 3ch model
    optimizer = optim.AdamW([
        {"params": [p for n,p in m3.named_parameters() if "classifier" not in n], "lr":BASE_LR_BACKBONE},
        {"params": m3.model.classifier.parameters(), "lr":BASE_LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
    best_f1 = 0
    patience = 0

    for epoch in range(NUM_EPOCHS):
        print(f"\nFOLD {fold+1} – 3ch – EPOCH {epoch+1}")

        tr_loss, tr_f1 = train_one_epoch(m3, train_loader_3ch, optimizer, criterion)
        val_loss, val_f1, _, _ = evaluate_slides(
            m3, val_rows, TRAIN_IMG_DIR, TRAIN_MASK_DIR,
            num_tiles=VAL_TILES_PER_SLIDE, three_channel=True
        )
        print(f"Train F1={tr_f1:.4f}  Val F1={val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            patience = 0
            torch.save(m3.state_dict(), f"best_fold{fold}_3ch.pth")
        else:
            patience += 1
            if patience >= PATIENCE:
                break
        scheduler.step()

    fold_models_3ch.append(f"best_fold{fold}_3ch.pth")



# ============================================================
# TEST INFERENCE – ENSEMBLE OVER FOLDS + 3ch/4ch
# ============================================================

print("\n========== TEST INFERENCE ==========")

test_fnames = sorted(os.listdir(TEST_IMG_DIR))
test_df = pd.DataFrame({"sample_index": test_fnames})

all_logits = []

with torch.no_grad():
    for fname in tqdm(test_fnames):
        img_path = os.path.join(TEST_IMG_DIR, fname)
        mask_path = os.path.join(TEST_MASK_DIR, fname)

        img = load_image(img_path)
        mask = load_mask(mask_path, img.size)

        # Extract tiles
        bboxes = compute_ranked_tiles(
            img, mask,
            tile_size=TILE_SIZE,
            overlap=TILE_OVERLAP,
            max_tiles=TEST_TILES_PER_SLIDE,
            roi_pad=ROI_PAD
        )

        slide_logits_accum = []

        for fold in range(NUM_FOLDS):
            # Load models
            m4 = DenseNet121_4ch(num_classes).to(DEVICE)
            m3 = DenseNet121_3ch(num_classes).to(DEVICE)
            m4.load_state_dict(torch.load(f"best_fold{fold}_4ch.pth", map_location=DEVICE))
            m3.load_state_dict(torch.load(f"best_fold{fold}_3ch.pth", map_location=DEVICE))
            m4.eval()
            m3.eval()

            tile_logits_fold = []

            for (l,t,rgt,btm) in bboxes:
                tile_img = img.crop((l,t,rgt,btm))
                tile_mask = mask.crop((l,t,rgt,btm))

                x4 = transform_eval(tile_img, tile_mask, three_channel=False).unsqueeze(0).to(DEVICE)
                x3 = transform_eval(tile_img, tile_mask, three_channel=True).unsqueeze(0).to(DEVICE)

                logits4 = m4(x4)
                logits3 = m3(x3)

                tile_logits_fold.append( (logits4 + logits3) / 2 )

            fold_logits = torch.mean(torch.stack(tile_logits_fold), dim=0)
            slide_logits_accum.append(fold_logits)

        slide_logits = torch.mean(torch.stack(slide_logits_accum), dim=0)
        all_logits.append(slide_logits.cpu().numpy())

all_logits = np.vstack(all_logits)
preds = np.argmax(all_logits, axis=1)
labels_str = label_encoder.inverse_transform(preds)

submission = pd.DataFrame({
    "sample_index": test_fnames,
    "label": labels_str
})

submission.to_csv("submission.csv", index=False)
print("Submission saved!")



===== FOLD 1/3 =====
Precomputing tiles...


100%|██████████| 387/387 [00:23<00:00, 16.68it/s]


Total tiles: 1161
Precomputing tiles...


100%|██████████| 387/387 [00:22<00:00, 17.03it/s]


Total tiles: 1161

FOLD 1 – 4ch – EPOCH 1


...


Validating: 100%|██████████| 194/194 [00:45<00:00,  4.27it/s]

Train F1=0.2402  Val F1=0.2445

FOLD 1 – 3ch – EPOCH 2



Validating: 100%|██████████| 194/194 [00:45<00:00,  4.28it/s]

Train F1=0.3027  Val F1=0.2661

FOLD 1 – 3ch – EPOCH 3



Validating:  20%|██        | 39/194 [00:09<00:36,  4.21it/s]